# ML-02 — Research Question and Provisional Lane

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane (or freestyle) and why

*Name your lane — or say 'freestyle' and describe your own question. One short paragraph: why this one?*

**Lane 2 — Refresh / Content Opportunity Scoring.**

I picked this lane for three reasons, in order of how much they actually decided it.

First, the decision is already concrete. "Which page does an editor open first on Monday?" is a real decision with a real person attached, and I can name the action they take when they get there. Some lanes need me to invent the decision after the analysis; this one starts with it.

Second, the starter dataset is this lane's default dataset, not a stand-in. I can do seven weeks of honest work without waiting on warehouse access, and move to the warehouse in Week 3 for stronger time-window labels because I *want* better labels — not because I was blocked.

Third, and the reason I'd defend the choice: the lane ships with a baseline that is already measurably bad. `scripts/02_baseline_score.py` is a transparent hand rule, and when I ran it, it scored **0.240 Precision@50 against a 0.391 holdout base rate** — worse than ordering the queue at random. That is a rare and useful starting position. I am not hunting for a problem to justify a model; the gap is measured, on data I already have, and I have a number to beat from day one.

*Provisional under the Week-4 rule.* What would move me: if the signal turns out to be mostly one confounder (page age, say), the honest version of this project is a signal analysis (Lane 1), not a scoring model.


## 2. The question: decision, action, cost of a wrong call

*What decision does your work improve? Who acts on it? What does a wrong recommendation cost?*

### The question

> **Which pages should be reviewed first for refresh, expansion, protection, pruning, or monitoring?**

### The decision it improves

Not "predict decline" — the **ordering of a weekly review queue**. An editor has a fixed number of hours and a backlog far larger than those hours (Section 3 sizes it). They will review *some* pages regardless. The only thing my work changes is **which ones, and in what order**. That framing matters: it means I am judged at the top of the list, not on overall accuracy.

### Who acts, and what they do

A **content/SEO editor** working across several client sites. They open the ranked queue, take the top N their week allows, and for each one pick a single action: **refresh, expand, protect, prune, or monitor**. For that to be usable, each row needs a *reason code* — an editor will not act on a bare score, and shouldn't. A score with no stated reason is unauditable, and "hiding the reason behind a high score" is one of the lane's named failure modes.

### What a wrong call costs — and why the two errors are not equal

| Error | What happens | Cost |
|---|---|---|
| **False positive** — ranked high, didn't need work | Editor spends time on a healthy page; may edit a page that was fine and lose the ranking it had | ~1–3 editor hours, plus real edit risk. **Visible and recoverable.** |
| **False negative** — declining, high-impression page never surfaces | Decline compounds for another quarter on a page that was still getting traffic | Larger, and **invisible** — nobody finds out what the queue never showed them |

The false negative is worse *and* it is the one that never shows up in a demo. So Precision@K alone will flatter me: it only grades what I did surface. I will pair it with a check on what the queue **misses** among high-impression declining pages, and hand-review my top 20 (a method the lane guide explicitly asks for).

### Why data/ML at all, rather than an if-statement

This is the question that should kill most projects, so I checked it rather than asserting it. The repo already *contains* the if-statement, and I measured it — **0.240 P@50 against a 0.391 base rate, worse than random ordering**. Meanwhile per-client decline rates run from 0.00 to 0.94 (Section 3), so no single global threshold like "stale AND visible" can be right for all 32 clients at once.

That combination — a hand rule that is measurably worse than random, plus a signal whose scale shifts per client — is the specific case where ML earns its place: the pattern is real but too tangled to hand-write. If the hand rule had scored 0.60, the honest answer would have been "tune the rule, skip the model."

### Task type and metric, named before training

| | |
|---|---|
| **Task type** | Ranking / scoring (from `framing-ml-problems`: "which ones first?" → ranking) |
| **Target** | `is_declining_label`, derived from the observed `trend_direction` bucket |
| **Primary metric** | Precision@50 — one editor-week of queue |
| **Guardrail metrics** | Recall among high-impression declining pages; average precision; hand-review of top 20 |
| **Validation** | Client-holdout (`GroupShuffleSplit` on `client_id`), repeated over multiple splits — a single split moved my Week-2 result by 0.14, so one split is not a result |
| **Never features** | `trend_direction`, `trend_pct` — the label is derived from them |

**One honesty flag I am carrying forward:** my target is *observed* in the sense that it is measured from traffic, but it arrives pre-bucketed as `trend_direction`. I am therefore predicting **a recorded trend bucket**, not future traffic. Week 3 on the warehouse daily table is where I can define a real forward-looking time-window label and check whether this framing survives.


## 3. Quick look at the data (2-3 real numbers)

*Load the starter CSV below and show 2-3 real numbers that make your lane look worth the next 7 weeks.*

In [1]:
import json, os
from pathlib import Path
import pandas as pd

# Work from the repo root whether this is opened from work/notebooks/ or the root.
if Path.cwd().name == "notebooks":
    os.chdir("../..")

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")
df["declining"] = df["trend_direction"].str.lower().eq("down")

print(f"{len(df):,} pages | {df['client_id'].nunique()} pseudonymized clients\n")

# --- Number 1: is the backlog big enough that ORDERING is the real problem? -------
# "Visible" = still earning impressions, so a refresh could plausibly matter.
backlog = df[df["declining"] & (df["impressions_90d"] >= 500)]
WEEKLY_CAPACITY = 50  # one editor-week of review, and my Precision@K choice
print("1. THE BACKLOG IS THE PROBLEM")
print(f"   declining AND impressions_90d >= 500 : {len(backlog):,} pages")
print(f"   at {WEEKLY_CAPACITY} reviews/week      : {len(backlog) / WEEKLY_CAPACITY:,.0f} weeks of queue")
print("   -> nobody clears this list. Only the ORDER is actionable.\n")

# --- Number 2: the existing hand rule is worse than random at the top ------------
# These come from `python scripts/run_all.py` (gitignored, so guard for a fresh clone).
print("2. THE SHIPPED HAND RULE IS WORSE THAN RANDOM AT K=50")
results_path = Path("outputs/model_results.json")
if results_path.exists():
    r = json.loads(results_path.read_text())
    preds = pd.read_csv("data/processed/model_predictions.csv")
    # The holdout base rate is the honest comparison, NOT the global 0.542.
    holdout_rate = preds.loc[preds["split"] == "test", "is_declining_label"].mean()
    print(f"   hand-rule baseline Precision@50 : {r['baseline']['baseline_precision_at_50']:.3f}")
    print(f"   holdout base rate (random order): {holdout_rate:.3f}   <- the bar")
    print(f"   random forest Precision@50      : {r['models']['random_forest']['precision_at_50']:.3f}")
    print("   -> the transparent rule loses to random. That gap is the project.\n")
else:
    print("   run `python scripts/run_all.py` first (outputs/ is gitignored).\n")

# --- Number 3: the signal is not on one global scale ----------------------------
per_client = df.groupby("client_id")["declining"].agg(["size", "mean"])
per_client = per_client[per_client["size"] >= 50]  # ignore tiny clients
print("3. 'DECLINING' MEANS DIFFERENT THINGS PER CLIENT")
print(f"   {len(per_client)} clients with >= 50 pages")
print(f"   decline rate min {per_client['mean'].min():.3f} | "
      f"median {per_client['mean'].median():.3f} | "
      f"max {per_client['mean'].max():.3f} | sd {per_client['mean'].std():.3f}")
print("   -> one global threshold cannot fit all clients. This is also why my")
print("      train/test base rates differ (0.555 vs 0.391) under a client holdout,")
print("      and why validation must group by client_id.")


30,000 pages | 32 pseudonymized clients

1. THE BACKLOG IS THE PROBLEM
   declining AND impressions_90d >= 500 : 9,961 pages
   at 50 reviews/week      : 199 weeks of queue
   -> nobody clears this list. Only the ORDER is actionable.

2. THE SHIPPED HAND RULE IS WORSE THAN RANDOM AT K=50
   hand-rule baseline Precision@50 : 0.240
   holdout base rate (random order): 0.391   <- the bar
   random forest Precision@50      : 0.680
   -> the transparent rule loses to random. That gap is the project.

3. 'DECLINING' MEANS DIFFERENT THINGS PER CLIENT
   25 clients with >= 50 pages
   decline rate min 0.000 | median 0.549 | max 0.937 | sd 0.227
   -> one global threshold cannot fit all clients. This is also why my
      train/test base rates differ (0.555 vs 0.391) under a client holdout,
      and why validation must group by client_id.


## 4. Careful words: what I can and can't claim

*Write what your work will be able to say (observed, directional, decision-support) — and what it never will (causal proof, 'predicting Google').*

### What I will be able to say

**Observed / measured** — statements about this dataset, with the sample attached:

> "In a 90-day anonymized snapshot of 30,000 pages across 32 pseudonymized clients, 9,961 pages were both declining and still earning 500+ impressions."

**Directional** — comparisons that held up under repetition, quoted with their spread:

> "Under repeated client-holdout splits, the learned ranking placed more genuinely declining pages in its top 50 than the hand-written rule did — roughly 0.67 vs 0.58 Precision@50, with a standard deviation near 0.08 across splits."

**Decision-support** — the output framed as a starting point for a person:

> "This queue suggests which pages an editor might open first. Each row carries a reason code so the editor can disagree with it."

### What I will never say

**No causal claims.** I have no intervention data — nobody randomly refreshed half these pages for me. I can observe that declining pages tend to share certain properties. I cannot say *"refreshing these pages will recover traffic."* That sentence needs an experiment I do not have, and it is the exact mistake the lane guide names: treating "declining" as a guarantee that a refresh will pay off.

**No claims about Google.** I am not reverse-engineering a ranking algorithm. I am describing patterns in one agency's measured performance data. "I predicted Google's algorithm" is never a sentence I get to write.

**No pretending my label is the future.** `is_declining_label` comes from `trend_direction`, a bucket already computed in the shipped data. I am predicting **a recorded trend bucket**, not future traffic. Every claim I make inherits that limitation, and I will state it rather than let a reader assume otherwise.

**No false precision.** My Week-2 numbers moved by up to 0.14 between client splits. So I report "roughly 0.67, ±0.08 across splits," never "0.672." Any difference smaller than the split-to-split spread is not a difference.

**Nothing identifying.** `content_id` and `client_id` are pseudonyms — they are for grouping and splitting, never features, and never published. No client names, domains, URLs, page titles, or raw queries appear in this notebook or anything built from it.

### The three risks most likely to make me wrong

1. **Confounding.** If page age drives both "stale" and "declining," my model may be an age detector wearing a costume. Week 4's signal audit has to separate these — and if it can't, Lane 1 is the honest project.
2. **Client mix.** Decline rates run 0.00 to 0.94 per client. A model that looks good may just be ranking *clients*, not pages. Grouped validation is the check, and it is why I never split on rows.
3. **Label drift.** The 90-day window is one snapshot of one period. Nothing here establishes that the pattern holds in a different quarter, and I won't imply that it does.


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere — only pseudonymous `client_id` counts, never listed individually
- [x] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.

**Note on the last box:** it stays unticked until the repo exists. This folder is currently an unzipped download, not a git repo — creating it from the template and pushing is the remaining step.

### Where this goes next

| Assignment | Notebook | What this framing commits me to |
|---|---|---|
| ML-04 | `w03_data_contract.ipynb` | Write the contract that bans `trend_direction` / `trend_pct` as features, and handle `avg_position == 0` as missing rather than rank zero |
| ML-06 | `w04_signal_audit.ipynb` | Test risk #1 — is this an age detector? Separate staleness from age |
| ML-07 | `w04_baseline_score.ipynb` | Rebuild the hand rule per-client instead of globally, and see whether that alone clears 0.391 |
| ML-08/09 | `w05_model.ipynb`, `w06_validation_audit.ipynb` | Repeated client-holdout, report spread not point estimates, plus the missed-page check Precision@K hides |
